# Pipeline Itapeva — Contaminação por *Cryptosporidium* (nov/2026)

**Global Solution 2026.2 — Big Data & Analytics — FIAP**

Notebook para execução passo a passo do pipeline Medallion (Bronze → Silver → Gold)
usando **Pandas + DuckDB (SQL)** localmente, sem depender de Java/Spark.

Para a versão PySpark em Databricks Community Edition, veja `pipeline_itapeva.py`.

## Execução

Antes de rodar este notebook, garanta que os dados sintéticos foram gerados:
```
python -m src.utils.gera_dados_sinteticos
```

Depois execute célula a célula (Shift+Enter).

## Passo 0 — Imports e configuração

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRONZE = ROOT / 'data' / 'bronze'
SILVER = ROOT / 'data' / 'silver'
GOLD   = ROOT / 'data' / 'gold'
SILVER.mkdir(parents=True, exist_ok=True)
GOLD.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(':memory:')
print('ROOT:', ROOT)
print('DuckDB:', duckdb.__version__)

## Passo 1 — Bronze: leitura dos dados brutos

Fontes carregadas: SISAGUA, SIH-SUS, INMET, SNIS, IBGE (bairros) e dados da crise 2026
(casos por bairro, pontos de distribuição, laudos IAL).

In [ ]:
bronze = {
    'bronze_sisagua': pd.read_csv(BRONZE / 'sisagua' / 'sisagua_itapeva_2015_2026.csv'),
    'bronze_sih':     pd.read_csv(BRONZE / 'sih_sus' / 'sih_itapeva_2018_2026.csv'),
    'bronze_inmet':   pd.read_csv(BRONZE / 'inmet'   / 'inmet_a714_2020_2026.csv'),
    'bronze_snis':    pd.read_csv(BRONZE / 'snis'    / 'snis_itapeva.csv'),
    'bronze_bairros': pd.read_csv(BRONZE / 'ibge'    / 'bairros_itapeva.csv'),
    'bronze_casos':   pd.read_csv(BRONZE / 'crise_2026' / 'casos_por_bairro_diario.csv'),
    'bronze_pontos':  pd.read_csv(BRONZE / 'crise_2026' / 'pontos_distribuicao.csv'),
    'bronze_laudos':  pd.read_csv(BRONZE / 'crise_2026' / 'laudos_ial.csv'),
}
for nome, df in bronze.items():
    con.register(nome, df)
    print(f'{nome:>18}: {len(df):>7} linhas')

In [ ]:
bronze['bronze_sisagua'].head()

## Passo 2 — Silver: padronização e Feature Engineering (SQL)

- Tipagem de colunas (datas, numéricos)
- `flag_nao_conforme` na qualidade da água (regras Port. MS 888/2021)
- `grupo_cid` consolidando CIDs A0x em Gastroenterites
- **Feature Engineering de clima**: `chuva_acum_7d` e `chuva_acum_14d`
  (janela móvel) — antecipa pico de turbidez bruta

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE silver_qualidade_agua AS
SELECT
    CAST(cod_ibge AS INTEGER)              AS cod_ibge,
    municipio, uf, eta,
    CAST(data_coleta AS DATE)              AS data_coleta,
    EXTRACT(year  FROM CAST(data_coleta AS DATE)) AS ano,
    EXTRACT(month FROM CAST(data_coleta AS DATE)) AS mes,
    CAST(cloro_residual_mg_l AS DOUBLE)    AS cloro_residual_mg_l,
    CAST(turbidez_ntu AS DOUBLE)           AS turbidez_ntu,
    CAST(coliformes_totais_ufc AS INTEGER) AS coliformes_totais_ufc,
    CAST(ecoli_ufc AS INTEGER)             AS ecoli_ufc,
    conformidade,
    CASE
        WHEN cloro_residual_mg_l < 0.2 THEN 1
        WHEN turbidez_ntu > 5          THEN 1
        WHEN ecoli_ufc > 0             THEN 1
        ELSE 0
    END AS flag_nao_conforme
FROM bronze_sisagua
""")

con.execute("""
CREATE OR REPLACE TABLE silver_saude AS
SELECT
    CAST(cod_ibge_municipio AS INTEGER) AS cod_ibge,
    municipio, uf,
    CAST(mes_ano || '-01' AS DATE) AS data_ref,
    mes_ano, cid10,
    CASE
        WHEN cid10 LIKE 'A0%' THEN 'Gastroenterites'
        WHEN cid10 = 'B15'    THEN 'Hepatite A'
        ELSE 'Outros' END AS grupo_cid,
    CAST(n_internacoes AS INTEGER) AS n_internacoes,
    CAST(obitos AS INTEGER) AS obitos
FROM bronze_sih
""")

con.execute("""
CREATE OR REPLACE TABLE silver_clima AS
SELECT estacao, municipio, uf,
       CAST(data AS DATE) AS data,
       CAST(precipitacao_mm AS DOUBLE) AS precipitacao_mm,
       CAST(temperatura_c AS DOUBLE)   AS temperatura_c,
       CAST(umidade_pct AS DOUBLE)     AS umidade_pct
FROM bronze_inmet
""")

con.execute("""
CREATE OR REPLACE TABLE silver_clima_feat AS
SELECT *,
       SUM(precipitacao_mm) OVER (ORDER BY data ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  AS chuva_acum_7d,
       SUM(precipitacao_mm) OVER (ORDER BY data ROWS BETWEEN 13 PRECEDING AND CURRENT ROW) AS chuva_acum_14d
FROM silver_clima
""")

con.execute("""
CREATE OR REPLACE TABLE silver_bairros_dim AS
SELECT CAST(cod_ibge AS INTEGER) AS cod_ibge,
       bairro, eta, classe_renda,
       CAST(populacao AS INTEGER) AS populacao,
       CAST(lat AS DOUBLE) AS lat, CAST(lon AS DOUBLE) AS lon
FROM bronze_bairros
""")

for t in ['silver_qualidade_agua','silver_saude','silver_clima','silver_clima_feat','silver_bairros_dim']:
    n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:>22}: {n:>6} linhas')

# salva em parquet
for t in ['silver_qualidade_agua','silver_saude','silver_clima','silver_bairros_dim']:
    con.execute(f"COPY {t} TO '{SILVER/(t.replace('silver_','')+'.parquet')}' (FORMAT PARQUET)")
print('\nSilver gravada em', SILVER)

## Passo 3 — Gold: tabelas analíticas (SQL)

- `gold_fato_crise`: fato diário por bairro integrando casos + clima + socioeconômico
- `gold_indicador_risco_bairro`: ranking de risco por bairro desde 20/out/2026
- `gold_ranking_eta`: comparativo Pilão × Aranhas
- `gold_linha_tempo_crise`: casos, internações e chuva por semana

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE gold_fato_crise AS
WITH casos AS (
    SELECT CAST(data AS DATE) AS data, bairro, eta,
           CAST(casos_notificados AS INTEGER) AS casos_notificados,
           CAST(internacoes AS INTEGER)       AS internacoes
    FROM bronze_casos
)
SELECT c.data, c.bairro, c.eta, b.populacao, b.classe_renda,
       c.casos_notificados, c.internacoes,
       ROUND(c.internacoes * 100000.0 / b.populacao, 2) AS taxa_internacao_100k,
       cf.precipitacao_mm, cf.chuva_acum_7d, cf.chuva_acum_14d
FROM casos c
JOIN silver_bairros_dim b ON c.bairro = b.bairro
LEFT JOIN silver_clima_feat cf ON c.data = cf.data
""")

con.execute("""
CREATE OR REPLACE TABLE gold_indicador_risco_bairro AS
WITH agg AS (
    SELECT bairro, eta, classe_renda, MAX(populacao) AS populacao,
           SUM(casos_notificados) AS total_casos,
           SUM(internacoes)       AS total_internacoes
    FROM gold_fato_crise WHERE data >= DATE '2026-10-20'
    GROUP BY bairro, eta, classe_renda
)
SELECT bairro, eta, classe_renda, populacao,
       total_casos, total_internacoes,
       ROUND(total_internacoes * 100000.0 / populacao, 2) AS taxa_internacao_100k,
       CASE
           WHEN total_internacoes * 100000.0 / populacao > 300 THEN 'CRITICO'
           WHEN total_internacoes * 100000.0 / populacao > 200 THEN 'ALTO'
           WHEN total_internacoes * 100000.0 / populacao > 100 THEN 'MEDIO'
           ELSE 'BAIXO' END AS nivel_risco
FROM agg ORDER BY taxa_internacao_100k DESC
""")

con.execute("""
CREATE OR REPLACE TABLE gold_ranking_eta AS
SELECT eta, SUM(casos_notificados) AS casos_total,
       SUM(internacoes) AS internacoes_total,
       SUM(populacao) AS pop_total,
       ROUND(SUM(internacoes) * 100000.0 / SUM(populacao), 2) AS taxa_int_100k
FROM gold_fato_crise WHERE data >= DATE '2026-10-20'
GROUP BY eta ORDER BY taxa_int_100k DESC
""")

con.execute("""
CREATE OR REPLACE TABLE gold_linha_tempo_crise AS
WITH sc AS (SELECT DATE_TRUNC('week', data) AS semana, SUM(casos_notificados) AS casos, SUM(internacoes) AS internacoes FROM gold_fato_crise GROUP BY DATE_TRUNC('week', data)),
     scc AS (SELECT DATE_TRUNC('week', data) AS semana, SUM(precipitacao_mm) AS chuva_semanal FROM silver_clima WHERE data >= DATE '2026-09-01' GROUP BY DATE_TRUNC('week', data))
SELECT sc.semana, sc.casos, sc.internacoes, scc.chuva_semanal
FROM sc LEFT JOIN scc ON sc.semana = scc.semana ORDER BY sc.semana
""")

for t in ['gold_fato_crise','gold_indicador_risco_bairro','gold_ranking_eta','gold_linha_tempo_crise']:
    n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'{t:>32}: {n:>5} linhas')
    con.execute(f"COPY {t} TO '{GOLD/(t.replace('gold_','')+'.parquet')}' (FORMAT PARQUET)")

## Passo 4 — Analytics: respostas às perguntas do enunciado

In [ ]:
# P1. Como evoluiu a qualidade da agua antes da crise?
con.execute("""
SELECT ano, mes, eta,
       ROUND(AVG(turbidez_ntu), 2) AS turbidez,
       ROUND(AVG(cloro_residual_mg_l), 2) AS cloro,
       SUM(flag_nao_conforme) AS n_nao_conformes
FROM silver_qualidade_agua
WHERE data_coleta >= DATE '2026-06-01'
GROUP BY ano, mes, eta ORDER BY ano, mes, eta
""").df()

In [ ]:
# P2. Ranking de risco por bairro
risco = con.execute('SELECT * FROM gold_indicador_risco_bairro').df()
risco

In [ ]:
# P3. Pilao vs Aranhas
con.execute('SELECT * FROM gold_ranking_eta').df()

In [ ]:
# P4. Linha do tempo
tempo = con.execute('SELECT * FROM gold_linha_tempo_crise').df()
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(tempo['semana'], tempo['casos'], marker='o', color='red', label='Casos')
ax1.plot(tempo['semana'], tempo['internacoes'], marker='s', color='orange', label='Internacoes')
ax1.set_ylabel('Casos / Internacoes')
ax1.legend(loc='upper left')
ax2 = ax1.twinx()
ax2.bar(tempo['semana'], tempo['chuva_semanal'], alpha=0.3, color='blue', width=5, label='Chuva (mm)')
ax2.set_ylabel('Chuva semanal (mm)')
plt.title('Linha do tempo da crise - casos, internacoes e chuva semanal')
plt.tight_layout()
plt.show()

## Passo 5 — Machine Learning

- **KMeans** para clusterizar bairros por perfil (renda, população, taxa de internação)
- **RandomForest** para prever internações diárias a partir de chuva acumulada e casos defasados

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = risco[['populacao', 'total_casos', 'taxa_internacao_100k']].fillna(0)
X_s = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_s)
risco['cluster'] = km.labels_
risco[['bairro', 'eta', 'classe_renda', 'nivel_risco', 'cluster']]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

fato = con.execute('SELECT * FROM gold_fato_crise').df().sort_values(['bairro', 'data'])
fato['casos_lag1'] = fato.groupby('bairro')['casos_notificados'].shift(1).fillna(0)
fato['chuva_acum_7d'] = fato['chuva_acum_7d'].fillna(0)

feat = ['chuva_acum_7d', 'casos_lag1', 'populacao']
X, y = fato[feat], fato['internacoes']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(Xtr, ytr)
yp = rf.predict(Xte)
print(f'MAE: {mean_absolute_error(yte, yp):.3f}')
print(f'R^2: {r2_score(yte, yp):.3f}')
print('\nImportancia das features:')
for f, imp in zip(feat, rf.feature_importances_):
    print(f'  {f:<16} {imp:.3f}')

## Passo 6 — Resumo executivo

Síntese dos resultados que alimentam o relatório executivo (`docs/09_relatorio_executivo_template.md`).

In [ ]:
tot = con.execute('SELECT SUM(casos_notificados), SUM(internacoes) FROM gold_fato_crise').fetchone()
print(f'Total de casos notificados:  {tot[0]:,}')
print(f'Total de internacoes:        {tot[1]:,}')
print('\nBairros em nivel CRITICO:')
risco[risco['nivel_risco']=='CRITICO'][['bairro','eta','taxa_internacao_100k']]

## Próximos passos

1. Abra o dashboard: `streamlit run dashboard/app.py`
2. Para rodar no **Databricks Community**, use `pipeline_itapeva.py` (PySpark)
3. Para substituir os dados sintéticos pelos reais, veja os scripts em `src/bronze/`